# Scammer LLM Fine-Tuning

1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Upload `training_data.jsonl` when prompted

In [ ]:
!pip install -q --upgrade transformers peft datasets trl bitsandbytes accelerate torchao

In [ ]:
from google.colab import files
import os
os.makedirs("ml/data", exist_ok=True)
os.makedirs("ml/models/scammer-llm", exist_ok=True)
print("Upload training_data.jsonl:")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"ml/data/{name}")
import json
with open("ml/data/training_data.jsonl") as f:
    lines = f.readlines()
print(f"Loaded {len(lines)} examples")

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "Qwen/Qwen2-0.5B"

# Dataset
dataset = load_dataset("json", data_files="ml/data/training_data.jsonl", split="train")
def fmt(ex):
    msgs = ex.get("conversations", ex.get("messages", []))
    return {"text": "\n".join(f"<|{m.get('role','user')}|>\n{m.get('content','')}<|end|>" for m in msgs) + "\n"}
dataset = dataset.map(fmt)
print(f"Dataset: {len(dataset)} examples")

# Configs
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
peft_config = LoraConfig(r=32, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Model
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Train
args = SFTConfig(output_dir="ml/models/scammer-llm", num_train_epochs=3, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, weight_decay=0.01, warmup_steps=50, lr_scheduler_type="cosine", logging_steps=10, save_strategy="epoch", fp16=False, bf16=False, report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=dataset)
print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
trainer.save_model("ml/models/scammer-llm")
tokenizer.save_pretrained("ml/models/scammer-llm")
print("Model saved!")

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
import shutil, glob, os
from peft import PeftModel
from transformers import AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen2-0.5B"
MERGED = "ml/models/scammer-llm-merged"
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True)
m = PeftModel.from_pretrained(base, "ml/models/scammer-llm")
m = m.merge_and_unload()
m.save_pretrained(MERGED)

# Copy tokenizer from cache
cache = glob.glob(os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen2-0.5B/snapshots/*"))[0]
for f in ["tokenizer.json","tokenizer_config.json","vocab.json","merges.txt","generation_config.json"]:
    s = os.path.join(cache, f)
    if os.path.exists(s): shutil.copy(s, MERGED)
print(f"Merged: {os.listdir(MERGED)}")

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py /content/ml/models/scammer-llm-merged --outfile /content/ml/models/scammer-llm/model.gguf --outtype q8_0
import os
p = "/content/ml/models/scammer-llm/model.gguf"
print(f"Size: {os.path.getsize(p)/1024/1024:.1f} MB" if os.path.exists(p) else "ERROR")

In [ ]:
from google.colab import files
files.download("/content/ml/models/scammer-llm/model.gguf")

## Done

```bash
cp ~/Downloads/model.gguf ml/models/scammer-llm/model.gguf
cd ml/server && python main.py
```